# Salida 3D: del corte al volumen (PENGWIN)

Este cuaderno completa el tramo que pide el avance 3: **procesar todos los cortes de cada volumen, restaurar las máscaras a la geometría original del MHA, asociar instancias entre cortes, revisar fusiones, divisiones y fragmentos pequeños, y ajustar el umbral de asociación con `val`.**

Qué ya existía en `proyecto_pengwin.ipynb` y se reutiliza (se carga, no se copia): `letterbox_pair`, `restore_letterbox_label_mask` (23.9), el decoder con skips, `build_detection_region_priors`, `deteccion_grid.py` (grid + NMS propio) y `distancias_fragmentos.py`. Lo que hace este cuaderno de nuevo:

| Sección | Qué agrega |
|---|---|
| 3 | Inferencia sobre **todos** los cortes (detección → NMS → priors → decoder) y caché de las salidas crudas |
| 4 | Decodificación 2D con control de cortes activos (`det_gate`, `gate_min_run`) y recorte por cajas; restauración a la geometría del MHA |
| 5 | Asociación 3D (reemplaza a 23.10): criterio de solapamiento a elegir, saltos entre cortes, secciones en varios trozos y alertas de fusión/división |
| 6 | Limpieza 3D: tamaño mínimo y absorción de trozos pegados |
| 7 | Evaluación **por fragmento en 3D** contra la referencia (la unidad del challenge), con taxonomía de errores |
| 8 | Ajuste de todos los umbrales con `val` (criterio: F1 por fragmento) |
| 9 | Pruebas con errores provocados a propósito |
| 10 | Ejecución: techo con máscaras reales → línea base con el modelo → ajuste → revisión → distancias → archivos de salida |

**Cómo correrlo.** Guárdalo dentro de la carpeta del proyecto (la que tiene `proyecto_pengwin.ipynb`, `splits/` y `data_mha/`) y ejecuta todo (*Run All*). Las secciones 10.1 y 10.2 usan solo las máscaras reales de `val` y funcionan **sin pesos entrenados**. Desde 10.3 se necesita `evidencias/entrenamiento_multitarea/modelo_multitarea_final.pt` (el archivo que exporta el cuaderno de entrenamiento) o, si se corre en el mismo kernel que la sección 24, la variable `componentes`.


## 1. Configuración y piezas del equipo


In [ ]:
import ast
import gc
import json
import os
import sys
import time
import warnings
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import SimpleITK as sitk
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from scipy import ndimage
from scipy.optimize import linear_sum_assignment
from skimage.exposure import equalize_adapthist      # los usa enhance_bone_contrast (viene del cuaderno del proyecto)
from skimage.filters import threshold_otsu

warnings.filterwarnings("ignore", message="'mode' parameter is deprecated")   # Pillow 11: letterbox_pair usa mode=

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"            # precisión mixta solo en GPU (consigna 4.3)
IMAGE_SIZE, GRID_SIZE = 256, 16
np.random.seed(SEED); torch.manual_seed(SEED)


def find_project_root(marker="proyecto_pengwin.ipynb", max_up=3):
    """Carpeta del proyecto: la actual o una de sus 3 superiores que contenga proyecto_pengwin.ipynb.
    También se puede fijar con la variable de entorno PENGWIN_ROOT o escribiendo ROOT a mano."""
    start = Path(os.environ.get("PENGWIN_ROOT", Path.cwd())).resolve()
    for folder in [start, *list(start.parents)[:max_up]]:
        if (folder / marker).is_file():
            return folder
    raise FileNotFoundError(
        f"No encuentro {marker} desde {start} ni en sus {max_up} carpetas superiores. "
        f"Guarda/abre este cuaderno dentro de la carpeta del proyecto, o escribe ROOT = Path(r'C:\\ruta\\del\\proyecto').")


ROOT = find_project_root()
BASE_DIR = ROOT                      # lo usa resolve_repo_path (viene de proyecto_pengwin.ipynb)
EVIDENCE_DIR = ROOT / "evidencias"
OUT_3D_DIR = EVIDENCE_DIR / "salida_3d"
OUT_3D_DIR.mkdir(parents=True, exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Proyecto:", ROOT, "| dispositivo:", DEVICE, "| AMP:", AMP)


# ---------------------------------------------------------------------------------------------
# Cargar SIN copiar las definiciones que el equipo ya escribió en sus cuadernos (misma técnica que
# entrenamiento_multitarea.ipynb: se leen las funciones/clases del .ipynb y se ejecutan aquí).
# ---------------------------------------------------------------------------------------------
def _target_names(node):
    targets = node.targets if isinstance(node, ast.Assign) else [node.target] if isinstance(node, ast.AnnAssign) else []
    names = set()
    for target in targets:
        if isinstance(target, ast.Name):
            names.add(target.id)
        elif isinstance(target, (ast.Tuple, ast.List)):
            names.update(item.id for item in target.elts if isinstance(item, ast.Name))
    return names


def _load_existing_definitions(notebook_path, definition_names, value_names=()):
    notebook_path = Path(notebook_path)
    if not notebook_path.exists():
        raise FileNotFoundError(f"No encuentro el cuaderno del equipo: {notebook_path}")
    notebook = json.loads(notebook_path.read_text(encoding="utf-8"))
    wanted_defs, wanted_values = set(definition_names), set(value_names)
    snippets, found_defs, found_values = [], set(), set()
    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        source = "".join(cell.get("source", []))
        try:
            tree = ast.parse(source)
        except SyntaxError:
            continue
        for node in tree.body:
            name = getattr(node, "name", None)
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and name in wanted_defs and name not in found_defs:
                snippets.append(ast.get_source_segment(source, node)); found_defs.add(name)
            elif isinstance(node, (ast.Assign, ast.AnnAssign)):
                selected = _target_names(node) & wanted_values
                if selected and not selected.issubset(found_values):
                    snippets.append(ast.get_source_segment(source, node)); found_values.update(selected)
    missing_defs, missing_values = wanted_defs - found_defs, wanted_values - found_values
    if missing_defs or missing_values:
        raise RuntimeError(f"Faltan definiciones en {notebook_path.name}: "
                           f"funciones/clases={sorted(missing_defs)}, constantes={sorted(missing_values)}")
    exec("\n\n".join(snippets), globals())
    return len(snippets)


n_proj = _load_existing_definitions(
    ROOT / "proyecto_pengwin.ipynb",
    definition_names={
        "resolve_repo_path", "apply_hu_window", "enhance_bone_contrast", "instance_to_semantic", "boxes_from_semantic",
        "letterbox_pair", "FundidoraPC", "FundidoraPCPlus", "build_backbone",
        "build_instance_segmentation_targets", "build_detection_region_priors",
        "SegmentationConvBlock", "backbone_feature_pyramid", "SkipFusionBlock", "CenterOffsetSkipDecoder",
        "restore_letterbox_label_mask",
    },
    value_names={"CLASS_ID_TO_NAME", "TARGET_COLORS"},
)
try:
    from cbam_modulos import CBAM, CompuertaResidualCBAM
except ModuleNotFoundError:
    _load_existing_definitions(ROOT / "cbam.ipynb", {"ChannelAttention", "SpatialAttention", "CBAM", "CompuertaResidualCBAM"})
from deteccion_grid import (GridDetectionHead, NMS_CONFIG, decode_predictions, nms_por_clase, postprocess)
from distancias_fragmentos import measure_fragment_separations
print(f"proyecto_pengwin.ipynb: {n_proj} definiciones | cbam | deteccion_grid.py | distancias_fragmentos.py: OK")

# --- tabla de casos con su split (dataset.csv lo trae desde la sección 11 del cuaderno del proyecto) ---
df = pd.read_csv(ROOT / "splits" / "dataset.csv", dtype={"case_id": str})
assert "split" in df.columns, "splits/dataset.csv debe tener la columna 'split' (la crea la sección 11)"
CASES = {s: sorted(df.loc[df["split"] == s, "case_id"]) for s in ("train", "val", "test")}
print({s: len(c) for s, c in CASES.items()}, "| casos de val:", CASES["val"])

REGION_OF_CLASS = {1: "sacro", 2: "coxal_izquierdo", 3: "coxal_derecho"}


## 2. Parámetros

Hay cuatro bloques. **Los valores iniciales reproducen lo que ya hay en el proyecto** (23.10 y la sección 24), de modo que la primera fila de cada tabla es la línea base y se ve qué mejora cada decisión. Todos se ajustan con `val` en la sección 8; nada se ajusta con `test`.

- `CFG_DECODE`: qué cortes se segmentan y cómo se pasa de las salidas del decoder a instancias 2D. Un corte solo se segmenta si su mejor caja supera `det_gate` y pertenece a una racha de `gate_min_run` cortes seguidos. Esto responde a que el modelo se entrenó **solo con cortes positivos** (el manifiesto no tiene cortes vacíos): al procesar el volumen entero, los cortes sin hueso son una fuente de falsos positivos aislados.
- `CFG_LINK`: asociación entre cortes. `overlap_mode="min"` es el criterio de 23.10.
- `CFG_POST`: limpieza de fragmentos 3D (tamaño mínimo, confianza, absorción de trozos pegados).


In [ ]:
# Cuatro bloques de parámetros. Los valores iniciales reproducen lo que ya hay en el proyecto (23.10 y 24):
# así la primera fila de la tabla de ajuste es la línea base y se ve qué mejora cada decisión.
CFG_SLICE = dict(            # inferencia corte a corte (cabeza grid + NMS propio)
    det_score_min=0.05,      # candidatas con score < 0.05 se descartan antes del NMS (igual que 24.1)
    nms_iou=NMS_CONFIG["iou_thresh"],
    max_det_per_class=NMS_CONFIG["max_det_per_class"],
    batch_size=16,
)
CFG_DECODE = dict(           # de las salidas del decoder a instancias 2D (se ajusta con val)
    center_threshold=0.25,   # pico de centro mínimo
    min_center_distance=7,   # distancia mínima entre picos (px a 256 x 256)
    min_instance_pixels=1,   # tamaño mínimo de una instancia 2D (px a 256 x 256): 1 = no borra nada
    fallback_components=True,
    det_gate=0.05,           # un corte solo se segmenta si su mejor caja tiene score >= det_gate
    gate_min_run=1,          # ...y si pertenece a una racha de >= gate_min_run cortes consecutivos activos
    box_margin=None,         # None = sin recorte; k = descarta píxeles fuera de las cajas predichas de su clase (+k px)
)
CFG_LINK = dict(             # asociación de instancias entre cortes
    min_overlap=0.10,        # umbral de la sección 23.10
    overlap_mode="min",      # "min": inter/área menor (23.10) | "iou": inter/unión | "cur": inter/área del corte actual
    max_gap=0,               # cortes vacíos que se toleran entre dos apariciones del mismo fragmento
    many_to_one=False,       # True: dos manchas de un mismo corte pueden pertenecer al mismo fragmento 3D
)
CFG_POST = dict(             # limpieza de fragmentos 3D
    min_slices=1,            # cortes mínimos que debe abarcar un fragmento 3D
    min_voxels=1,            # vóxeles mínimos (en la geometría ORIGINAL del MHA)
    min_score=0.0,           # confianza media mínima (pico de centro)
    absorb_max_voxels=0,     # >0: fragmentos más chicos que esto y pegados a uno más grande de su clase se le unen
)


## 3. Modelo e inferencia sobre todos los cortes

`infer_case` recorre **todos los cortes** de un volumen con el mismo preprocesamiento del dataset (ventana HU → Otsu + CLAHE → letterbox 256 × 256), aplica la cabeza grid con el NMS propio (`nms_por_clase`, `max_det_per_class=1`), construye los priors de región con las **cajas predichas** (nunca las reales) y corre el decoder. Guarda las salidas crudas (clase semántica, logits de centro, offsets y cajas) para poder cambiar los umbrales de decodificación sin repetir la red.

`oracle_bundle` fabrica esas mismas salidas **a partir de la etiqueta real**. No es un modelo: sirve para medir el *techo* del tramo restaurar → asociar → medir. Si con salidas perfectas el resultado ya es malo, el problema no es la red.


In [ ]:
def build_components_from_export(path, device=DEVICE):
    """Reconstruye {backbone, cbam, det_head, seg_decoder, features_fn} desde modelo_multitarea_final.pt
    (el archivo que exporta entrenamiento_multitarea.ipynb: pesos + descripción de la arquitectura)."""
    payload = torch.load(path, map_location=device, weights_only=True)
    arch, state = payload["architecture"], payload["model_state_dict"]
    backbone = build_backbone(arch["backbone_name"], in_channels=1)
    cbam = None
    if arch["use_cbam"]:
        cbam = CompuertaResidualCBAM(CBAM(256)) if arch.get("use_gate", True) else CBAM(256)
    parts = {"backbone": backbone, "cbam": cbam, "det_head": GridDetectionHead(),
             "seg_decoder": CenterOffsetSkipDecoder(in_channels=256, image_channels=1, use_skips=arch.get("use_skips", True))}
    for name, module in parts.items():
        if module is None:
            continue
        sub = {k[len(name) + 1:]: v for k, v in state.items() if k.startswith(name + ".")}
        module.load_state_dict(sub)
        module.to(device).eval()
    return {**parts, "features_fn": backbone_feature_pyramid}, payload


def trunk_features(comp, x):
    """Pirámide [f1, f2, f3, f4]; la compuerta CBAM refina solo f4 (misma regla que MultiTaskNet.trunk)."""
    feats = list(comp["features_fn"](comp["backbone"], x))
    if comp.get("cbam") is not None:
        out = comp["cbam"](feats[-1])
        feats[-1] = out[0] if isinstance(out, tuple) else out
    return feats


def get_components():
    """1) Si este cuaderno corre en el mismo kernel que la sección 24 del proyecto, usa `componentes` (Miguel/Fabián).
       2) Si no, carga evidencias/entrenamiento_multitarea/modelo_multitarea_final.pt (exportado por el cuaderno de entrenamiento).
       3) Si no hay pesos, devuelve None: todo lo que no necesita el modelo (techo del pipeline, ajuste con máscaras reales) sigue funcionando."""
    if "componentes" in globals() and globals()["componentes"].get("seg_decoder") is not None:
        return globals()["componentes"], "componentes del kernel (sección 24)"
    export = EVIDENCE_DIR / "entrenamiento_multitarea" / "modelo_multitarea_final.pt"
    if export.is_file():
        comp, payload = build_components_from_export(export)
        sel = payload.get("selected_from", {})
        return comp, f"{export.name} (etapa {sel.get('stage')}, época {sel.get('epoch')})"
    return None, "sin pesos entrenados"


In [ ]:
def load_case_volumes(case_id, with_label=True, with_image=True):
    """Lee imagen (y etiqueta) del MHA de un caso en la geometría ORIGINAL: arrays (z, y, x) + header de SimpleITK.
    with_image=False (solo etiqueta) ahorra ~400 MB cuando no se necesita la tomografía; el header sale de la etiqueta."""
    row = df[df["case_id"] == case_id].iloc[0]
    volume, label, header = None, None, None
    if with_image:
        header = sitk.ReadImage(str(resolve_repo_path(row["image_path"])))
        volume = sitk.GetArrayFromImage(header).astype(np.float32)
    if with_label:
        label_itk = sitk.ReadImage(str(resolve_repo_path(row["label_path"])))
        label = sitk.GetArrayFromImage(label_itk).astype(np.int16)
        header = header if header is not None else label_itk
        if volume is not None:
            assert label.shape == volume.shape, f"caso {case_id}: imagen {volume.shape} y etiqueta {label.shape} no coinciden"
    return volume, label, header


def preprocess_slices(volume_zyx, z_indices, image_size=IMAGE_SIZE):
    """Mismo preprocesamiento que PengwinDetectionDataset: ventana HU -> Otsu+CLAHE -> letterbox a 256 x 256."""
    zeros = np.zeros(volume_zyx.shape[1:], dtype=np.int16)
    images, transform = [], None
    for z in z_indices:
        image = enhance_bone_contrast(apply_hu_window(volume_zyx[z]))
        image, _, transform = letterbox_pair(image, zeros, image_size)
        images.append(image)
    return np.stack(images)[:, None].astype(np.float32), transform


def empty_bundle(case_id, source, shape, spacing_xyz, transform):
    n_slices, size = shape[0], IMAGE_SIZE
    return {"case_id": case_id, "source": source, "shape": tuple(shape), "spacing_xyz": tuple(spacing_xyz),
            "transform": transform,
            "sem": np.zeros((n_slices, size, size), np.uint8),               # clase semántica por píxel (0..3)
            "ctr": np.full((n_slices, 3, size, size), -20.0, np.float16),    # logits de centro (3 regiones)
            "off": np.zeros((n_slices, 2, size, size), np.float16),          # offsets (dx, dy) normalizados
            "has_seg": np.zeros(n_slices, bool), "dets": [None] * n_slices}


@torch.no_grad()
def infer_case(comp, case_id, cfg=CFG_SLICE, device=DEVICE, amp=AMP, volume=None, image_itk=None, verbose=True):
    """TODOS los cortes del volumen -> cajas (grid + NMS) -> priors -> decoder. Guarda las salidas CRUDAS
    (semántica, centros, offsets, cajas) para poder cambiar los umbrales de decodificación sin repetir la red.
    Los cortes sin ninguna caja no pasan por el decoder (sin priors el decoder no tiene qué segmentar)."""
    if volume is None:
        volume, _, image_itk = load_case_volumes(case_id, with_label=False)
    n_slices = volume.shape[0]
    spacing = image_itk.GetSpacing() if image_itk is not None else (1.0, 1.0, 1.0)
    bundle = None
    for model in ("backbone", "cbam", "det_head", "seg_decoder"):
        if comp.get(model) is not None:
            comp[model].eval()
    t0 = time.time()
    for start in range(0, n_slices, cfg["batch_size"]):
        z_indices = list(range(start, min(n_slices, start + cfg["batch_size"])))
        images, transform = preprocess_slices(volume, z_indices)
        if bundle is None:
            bundle = empty_bundle(case_id, "model", volume.shape, spacing, transform)
        x = torch.from_numpy(images).to(device)
        with torch.autocast(device_type=device.type, enabled=amp):
            feats = trunk_features(comp, x)
            det_out = comp["det_head"](feats[-1])
        detections = postprocess(det_out.float(), score_thresh=cfg["det_score_min"], nms_fn=nms_por_clase,
                                 iou_thresh=cfg["nms_iou"], max_det_per_class=cfg["max_det_per_class"])
        with_boxes = [i for i, d in enumerate(detections) if len(d["boxes"]) > 0]
        for i, d in enumerate(detections):
            bundle["dets"][z_indices[i]] = {k: d[k].detach().cpu().numpy() for k in ("boxes", "scores", "labels")}
        if with_boxes:
            priors = build_detection_region_priors([detections[i] for i in with_boxes],
                                                   output_size=(IMAGE_SIZE, IMAGE_SIZE), device=device)
            with torch.autocast(device_type=device.type, enabled=amp):
                seg = comp["seg_decoder"](x[with_boxes], [None if f is None else f[with_boxes] for f in feats], priors)
            sem = seg["semantic_logits"].float().argmax(1).cpu().numpy().astype(np.uint8)
            ctr = seg["center_logits"].float().cpu().numpy().astype(np.float16)
            off = seg["offsets"].float().cpu().numpy().astype(np.float16)
            for k, i in enumerate(with_boxes):
                z = z_indices[i]
                bundle["sem"][z], bundle["ctr"][z], bundle["off"][z], bundle["has_seg"][z] = sem[k], ctr[k], off[k], True
        if verbose and (start // cfg["batch_size"]) % 5 == 0:
            print(f"  caso {case_id}: {z_indices[-1] + 1}/{n_slices} cortes ({time.time() - t0:.0f} s)")
    return bundle


def oracle_bundle(case_id, volume_label=None, image_itk=None):
    """Salidas del decoder 'perfectas' fabricadas desde la etiqueta real (centros gaussianos, offsets exactos, cajas reales).
    NO es un modelo: mide el TECHO del tramo restaurar -> asociar -> medir. Si aquí algo falla, el problema
    no es de la red sino de la geometría, del tamaño de los fragmentos o de los umbrales de asociación."""
    if volume_label is None:
        _, volume_label, image_itk = load_case_volumes(case_id, with_label=True, with_image=False)
    spacing = image_itk.GetSpacing() if image_itk is not None else (1.0, 1.0, 1.0)
    n_slices = volume_label.shape[0]
    zeros = np.zeros(volume_label.shape[1:], np.float32)
    bundle = None
    for z in range(n_slices):
        _, lab, transform = letterbox_pair(zeros, volume_label[z], IMAGE_SIZE)
        if bundle is None:
            bundle = empty_bundle(case_id, "gt", volume_label.shape, spacing, transform)
        if not lab.any():
            bundle["dets"][z] = {"boxes": np.zeros((0, 4), np.float32), "scores": np.zeros(0, np.float32),
                                 "labels": np.zeros(0, np.int64)}
            continue
        target = build_instance_segmentation_targets(lab)
        boxes, labels, _ = boxes_from_semantic(target["semantic"], min_pixels=1)
        bundle["dets"][z] = {"boxes": boxes, "scores": np.ones(len(boxes), np.float32), "labels": labels}
        centers = np.clip(target["centers"], 1e-4, 1 - 1e-4)
        bundle["sem"][z] = target["semantic"]
        bundle["ctr"][z] = np.log(centers / (1 - centers)).astype(np.float16)
        bundle["off"][z] = target["offsets"].astype(np.float16)
        bundle["has_seg"][z] = True
    return bundle


## 4. De las salidas a instancias 2D en la geometría del MHA

`decode_cached_slice` hace lo mismo que `decode_center_offset_instances` (23.x) con dos diferencias: (1) píxeles vecinos con **el mismo valor** de centro cuentan como un solo pico (con la versión anterior una meseta de valores iguales podía partir un fragmento en dos), y (2) acepta un recorte por cajas predichas. `decode_and_restore` aplica la decodificación a los cortes activos y devuelve cada máscara a la forma original `(y, x)` con `restore_letterbox_label_mask` (vecino más cercano, sin inventar IDs). Los IDs que salen aquí son **locales a cada corte**.


In [ ]:
def _logit(p):
    p = float(np.clip(p, 1e-6, 1 - 1e-6))
    return float(np.log(p / (1 - p)))


def _sigmoid(v):
    return 1.0 / (1.0 + np.exp(-np.asarray(v, dtype=np.float64)))


def _pick_peaks(heat_logits, threshold_logit, min_distance):
    """Máximos locales >= umbral. Los píxeles contiguos con el MISMO valor (mesetas) cuentan como un solo pico:
    con la versión de 23.x una meseta saturada producía varios picos vecinos y partía un fragmento en dos."""
    local_max = (heat_logits == ndimage.maximum_filter(heat_logits, size=2 * int(min_distance) + 1)) & (heat_logits >= threshold_logit)
    if not local_max.any():
        return np.zeros((0, 2), dtype=int)
    components, n_components = ndimage.label(local_max, structure=np.ones((3, 3)))
    return np.array(ndimage.maximum_position(heat_logits, components, range(1, n_components + 1)), dtype=int).reshape(-1, 2)


def boxes_allowed_mask(det, margin, size=IMAGE_SIZE):
    """[3, size, size] bool: píxeles dentro de alguna caja predicha de cada región (+ margen). Sin caja -> todo False."""
    allowed = np.zeros((3, size, size), dtype=bool)
    if det is None:
        return allowed
    for box, label in zip(det["boxes"], det["labels"]):
        x0, y0 = int(max(np.floor(box[0]) - margin, 0)), int(max(np.floor(box[1]) - margin, 0))
        x1, y1 = int(min(np.ceil(box[2]) + margin, size)), int(min(np.ceil(box[3]) + margin, size))
        allowed[int(label) - 1, y0:y1, x0:x1] = True
    return allowed


def decode_cached_slice(sem, ctr_logits, offsets, *, center_threshold, min_center_distance, min_instance_pixels,
                        fallback_components=True, allowed=None):
    """Misma lógica que decode_center_offset_instances (23.x) sobre las salidas guardadas, más:
    (1) meseta = un pico, (2) `allowed` opcional (recorte por cajas). Devuelve (instancias uint16, {id: clase}, {id: score})."""
    height, width = sem.shape
    instances = np.zeros((height, width), dtype=np.uint16)
    classes, scores = {}, {}
    next_id = 1
    threshold_logit = _logit(center_threshold)
    for class_id in (1, 2, 3):
        region = sem == class_id
        if allowed is not None:
            region = region & allowed[class_id - 1]
        if not region.any():
            continue
        heat = ctr_logits[class_id - 1].astype(np.float32)
        peaks = _pick_peaks(heat, threshold_logit, min_center_distance)
        if len(peaks) == 0:
            if fallback_components:            # región sin pico de centro: se separa por componentes conectados
                components, n_components = ndimage.label(region)
                for k in range(1, n_components + 1):
                    selected = components == k
                    if int(selected.sum()) < min_instance_pixels:
                        continue
                    instances[selected] = next_id
                    classes[next_id], scores[next_id] = class_id, float(_sigmoid(heat[selected].max()))
                    next_id += 1
            continue
        ys, xs = np.nonzero(region)
        voted_x = np.clip(xs + offsets[0, ys, xs].astype(np.float32) * width, 0, width - 1)
        voted_y = np.clip(ys + offsets[1, ys, xs].astype(np.float32) * height, 0, height - 1)
        d2 = (voted_x[:, None] - peaks[None, :, 1]) ** 2 + (voted_y[:, None] - peaks[None, :, 0]) ** 2
        owner = d2.argmin(axis=1)
        for k in range(len(peaks)):
            selected = owner == k
            if int(selected.sum()) < min_instance_pixels:
                continue
            instances[ys[selected], xs[selected]] = next_id
            classes[next_id], scores[next_id] = class_id, float(_sigmoid(heat[peaks[k, 0], peaks[k, 1]]))
            next_id += 1
    return instances, classes, scores


def active_slices(bundle, det_gate, gate_min_run):
    """Cortes que se segmentan: tienen salida del decoder, su mejor caja pasa `det_gate` y forman una racha
    de >= gate_min_run cortes seguidos (un hueso ocupa cortes contiguos; un corte aislado suele ser un falso positivo)."""
    n_slices = bundle["shape"][0]
    active = np.zeros(n_slices, dtype=bool)
    for z in range(n_slices):
        det = bundle["dets"][z]
        active[z] = bool(bundle["has_seg"][z] and det is not None and len(det["scores"]) and float(det["scores"].max()) >= det_gate)
    if gate_min_run > 1 and active.any():
        runs, n_runs = ndimage.label(active)
        for k in range(1, n_runs + 1):
            if int((runs == k).sum()) < gate_min_run:
                active[runs == k] = False
    return active


def decode_and_restore(bundle, dec_cfg, stride=1):
    """Cortes activos -> instancias 2D (256 x 256) -> geometría ORIGINAL del MHA (sin relleno del letterbox,
    vecino más cercano) -> volumen de IDs LOCALES por corte. `stride` > 1 submuestrea en el plano (solo para ajustar rápido)."""
    n_slices, height, width = bundle["shape"]
    out_h, out_w = (height + stride - 1) // stride, (width + stride - 1) // stride
    local = np.zeros((n_slices, out_h, out_w), dtype=np.uint16)
    classes_by_slice, scores_by_slice = [{} for _ in range(n_slices)], [{} for _ in range(n_slices)]
    active = active_slices(bundle, dec_cfg["det_gate"], dec_cfg["gate_min_run"])
    for z in np.flatnonzero(active):
        allowed = None if dec_cfg["box_margin"] is None else boxes_allowed_mask(bundle["dets"][z], dec_cfg["box_margin"])
        instances, classes, scores = decode_cached_slice(
            bundle["sem"][z], bundle["ctr"][z], bundle["off"][z],
            center_threshold=dec_cfg["center_threshold"], min_center_distance=dec_cfg["min_center_distance"],
            min_instance_pixels=dec_cfg["min_instance_pixels"], fallback_components=dec_cfg["fallback_components"],
            allowed=allowed)
        if not classes:
            continue
        restored = restore_letterbox_label_mask(instances, (height, width), bundle["transform"], IMAGE_SIZE)
        local[z] = restored[::stride, ::stride]
        classes_by_slice[z], scores_by_slice[z] = classes, scores
    return local, classes_by_slice, scores_by_slice


## 5. Asociación 3D

Reemplaza a `link_slice_instances_to_volume` (23.10), con la misma idea (asignación uno a uno óptima entre cortes consecutivos, solo dentro de la misma región) y cuatro cambios, todos controlados por parámetros:

1. **Criterio de solapamiento.** 23.10 divide la intersección entre el área *menor*. Con ese criterio un fragmento pequeño que aparece dentro de la sección del hueso grande tiene solapamiento 1.0 con él y puede **quedarse con su identidad** (el grande pasa a ser un fragmento nuevo). `iou` y `cur` no tienen ese problema; la prueba de la sección 9 lo reproduce.
2. **Saltos (`max_gap`).** Un fragmento puede faltar en uno o dos cortes y seguir siendo el mismo.
3. **Secciones en varios trozos (`many_to_one`).** Una sección 2D que sale en dos manchas puede seguir siendo un solo fragmento 3D.
4. **Alertas.** Se registran `merge_candidate` (una instancia toca ≥ 2 fragmentos previos) y `split_candidate` (un fragmento previo toca ≥ 2 instancias nuevas), ahora con los fragmentos a los que fue a parar cada trozo. **No se corrigen en silencio**: en 10.6 se contrastan con la referencia.


In [ ]:
def link_instances_3d(local_zyx, classes_by_slice, scores_by_slice, *, min_overlap=0.10, overlap_mode="min",
                      max_gap=0, many_to_one=False):
    """Asocia las instancias de cortes consecutivos en fragmentos 3D (IDs globales).

    * Solo se comparan instancias de la MISMA región. Asignación uno-a-uno óptima (húngaro) por región.
    * overlap_mode: "min" = inter / área menor (23.10) | "iou" = inter / unión | "cur" = inter / área en el corte actual.
    * max_gap: un fragmento puede 'saltar' hasta max_gap cortes sin aparecer y seguir siendo el mismo.
    * many_to_one: una mancha que no ganó la asignación pero se solapa >= min_overlap con un fragmento ya existente
      se le suma (así un fragmento cuya sección 2D sale en dos trozos no se parte en dos).
    * events: 'merge_candidate' (una instancia toca >= 2 fragmentos previos) y 'split_candidate' (un fragmento previo
      toca >= 2 instancias nuevas). Son alertas para revisar: no se corrigen en silencio.
    Devuelve (volumen uint16 de IDs globales, tabla de fragmentos, eventos)."""
    if overlap_mode not in ("min", "iou", "cur"):
        raise ValueError(f"overlap_mode desconocido: {overlap_mode}")
    n_slices, height, width = local_zyx.shape
    key = 1 << 16
    volume = np.zeros(local_zyx.shape, dtype=np.uint16)
    last_id = np.zeros((height, width), dtype=np.uint16)
    last_z = np.full((height, width), -10 ** 6, dtype=np.int32)
    track_class, info, events = {}, {}, []
    next_id = 1
    for z in range(n_slices):
        classes = classes_by_slice[z]
        if not classes:
            continue
        current = local_zyx[z]
        scores = scores_by_slice[z]
        recent = (last_id > 0) & ((z - last_z) <= max_gap + 1)
        previous = np.where(recent, last_id, 0).astype(np.uint16)
        pair_overlap = {}
        both = (previous > 0) & (current > 0)
        if both.any():
            pairs = previous[both].astype(np.int64) * key + current[both].astype(np.int64)
            unique_pairs, counts = np.unique(pairs, return_counts=True)
            area_prev = np.bincount(previous[previous > 0].astype(np.int64))
            area_cur = np.bincount(current[current > 0].astype(np.int64))
            for pair, inter in zip(unique_pairs, counts):
                g, l = int(pair // key), int(pair % key)
                if track_class.get(g) != classes.get(l):
                    continue
                ap, ac = int(area_prev[g]), int(area_cur[l])
                den = {"min": min(ap, ac), "iou": ap + ac - int(inter), "cur": ac}[overlap_mode]
                pair_overlap[(g, l)] = inter / den if den else 0.0
        mapping, pending_splits = {}, []
        for class_id in (1, 2, 3):
            tracks = sorted({g for (g, l) in pair_overlap if track_class[g] == class_id})
            locals_ = sorted(l for l, c in classes.items() if c == class_id)
            if not tracks or not locals_:
                continue
            matrix = np.zeros((len(tracks), len(locals_)))
            for (g, l), value in pair_overlap.items():
                if track_class[g] == class_id:
                    matrix[tracks.index(g), locals_.index(l)] = value
            linked = matrix >= min_overlap
            for i in np.flatnonzero(linked.sum(axis=1) > 1):
                pending_splits.append(len(events))
                events.append({"type": "split_candidate", "z": z, "class": class_id, "track": tracks[i],
                               "instances": [locals_[j] for j in np.flatnonzero(linked[i])],
                               "overlaps": [round(float(matrix[i, j]), 3) for j in np.flatnonzero(linked[i])]})
            for j in np.flatnonzero(linked.sum(axis=0) > 1):
                events.append({"type": "merge_candidate", "z": z, "class": class_id, "instance": locals_[j],
                               "tracks": [tracks[i] for i in np.flatnonzero(linked[:, j])],
                               "overlaps": [round(float(matrix[i, j]), 3) for i in np.flatnonzero(linked[:, j])]})
            rows, cols = linear_sum_assignment(-matrix)
            for i, j in zip(rows, cols):
                if matrix[i, j] >= min_overlap:
                    mapping[locals_[j]] = tracks[i]
            if many_to_one:
                for j, l in enumerate(locals_):
                    if l not in mapping and linked[:, j].any():
                        mapping[l] = tracks[int(matrix[:, j].argmax())]
        lookup = np.zeros(key, dtype=np.uint16)
        for l, class_id in classes.items():
            if l not in mapping:
                if next_id >= key:
                    raise RuntimeError("Más de 65 535 fragmentos 3D: la decodificación 2D está produciendo ruido. "
                                       "Sube min_instance_pixels o center_threshold antes de asociar.")
                mapping[l] = next_id
                track_class[next_id] = class_id
                info[next_id] = {"n_slices": 0, "n_inst": 0, "score_sum": 0.0, "z_min": z, "z_max": z, "last_z": -1}
                next_id += 1
            g = mapping[l]
            lookup[l] = g
            item = info[g]
            if item["last_z"] != z:
                item["n_slices"] += 1
                item["last_z"] = z
            item["score_sum"] += scores.get(l, 0.0)
            item["n_inst"] += 1
            item["z_max"] = z
        for index in pending_splits:                       # a qué fragmentos fueron a parar los trozos de cada división
            events[index]["to_tracks"] = [mapping[l] for l in events[index]["instances"]]
        volume[z] = lookup[current]
        painted = volume[z] > 0
        last_id[painted], last_z[painted] = volume[z][painted], z
    ids = np.array(sorted(track_class), dtype=int)
    counts = np.zeros(next_id, dtype=np.int64)
    for z in range(n_slices):
        if volume[z].any():
            counts += np.bincount(volume[z].ravel(), minlength=next_id)
    table = pd.DataFrame({
        "track_id": ids, "class": [track_class[i] for i in ids],
        "region": [REGION_OF_CLASS[track_class[i]] for i in ids],
        "n_voxels": [int(counts[i]) for i in ids], "n_slices": [info[i]["n_slices"] for i in ids],
        "z_min": [info[i]["z_min"] for i in ids], "z_max": [info[i]["z_max"] for i in ids],
        "mean_score": [info[i]["score_sum"] / max(info[i]["n_inst"], 1) for i in ids]})
    return volume, table, events


## 6. Limpieza 3D

El tamaño mínimo se expresa en **cortes** y en **vóxeles de la geometría original**. Ojo: es un filtro de doble filo. Un fragmento pequeño verdadero (puede haber astillas de muy pocos vóxeles) se parece a un falso positivo. Por eso los valores no se fijan a mano: se prueban con `val` y se mide el recall de los fragmentos pequeños (10.6). `absorb_max_voxels` une un trozo pequeño que **toca** a uno más grande de su región; corrige huesos grandes partidos, pero puede fundir un fragmento pequeño verdadero con el principal, que es justo el riesgo de fusión que menciona la consigna.


In [ ]:
def clean_tracks(volume, table, *, min_slices=1, min_voxels=1, min_score=0.0, absorb_max_voxels=0, stride=1):
    """Limpieza 3D de los fragmentos asociados. Devuelve (volumen con IDs consecutivos 1..N, tabla, bitácora).

    1. absorb_max_voxels > 0: un fragmento más chico que ese tamaño que TOCA (a <= 1 vóxel) a un fragmento más grande de su
       misma región se le une. Corrige que un hueso grande salga partido en trozos, pero también puede fundir un
       fragmento pequeño verdadero con el principal (el riesgo de fusión que pide la consigna): por eso se decide con val.
    2. Se eliminan los fragmentos con < min_slices cortes, < min_voxels vóxeles o confianza media < min_score.
    Los umbrales de tamaño se dan en vóxeles de la geometría ORIGINAL; con stride > 1 (ajuste rápido) se convierten."""
    volume = volume.copy()
    table = table.set_index("track_id").copy()
    voxel_scale = max(stride, 1) ** 2
    log = {"absorbidos": 0, "eliminados_por_cortes": 0, "eliminados_por_voxeles": 0, "eliminados_por_score": 0}
    table["absorbed_into"] = 0
    if absorb_max_voxels > 0 and len(table):
        objects = ndimage.find_objects(volume)
        limit = absorb_max_voxels / voxel_scale
        for track_id in table.sort_values("n_voxels").index:
            n_small = table.at[track_id, "n_voxels"]
            if n_small >= limit or n_small == 0 or objects[track_id - 1] is None:
                continue
            region = objects[track_id - 1]
            pad = tuple(slice(max(s.start - 2, 0), s.stop + 2) for s in region)
            crop = volume[pad]
            mask = crop == track_id
            if not mask.any():
                continue
            ring = ndimage.binary_dilation(mask, structure=np.ones((3, 3, 3), dtype=bool)) & ~mask
            neighbours, contact = np.unique(crop[ring][crop[ring] > 0], return_counts=True)
            candidates = [(int(c), int(n)) for n, c in zip(neighbours, contact)
                          if table.at[n, "class"] == table.at[track_id, "class"] and table.at[n, "n_voxels"] > n_small]
            if not candidates:
                continue
            target = max(candidates)[1]                    # el vecino con más contacto
            crop[mask] = target
            tgt_box = objects[target - 1]                  # la caja del destino crece: se mantiene al día
            objects[target - 1] = tuple(slice(min(a.start, b.start), max(a.stop, b.stop)) for a, b in zip(tgt_box, region))
            table.at[target, "n_voxels"] += n_small
            table.at[target, "z_min"] = min(table.at[target, "z_min"], table.at[track_id, "z_min"])
            table.at[target, "z_max"] = max(table.at[target, "z_max"], table.at[track_id, "z_max"])
            table.at[track_id, "n_voxels"] = 0
            table.at[track_id, "absorbed_into"] = target
            log["absorbidos"] += 1
        for target in sorted(set(table["absorbed_into"]) - {0}):       # cortes distintos que ocupa tras absorber
            table.at[target, "n_slices"] = int((volume == target).any(axis=(1, 2)).sum())
    alive = table["n_voxels"] > 0
    drop_slices = alive & (table["n_slices"] < min_slices)
    drop_voxels = alive & ~drop_slices & (table["n_voxels"] < min_voxels / voxel_scale)
    drop_score = alive & ~drop_slices & ~drop_voxels & (table["mean_score"] < min_score)
    log.update({"eliminados_por_cortes": int(drop_slices.sum()), "eliminados_por_voxeles": int(drop_voxels.sum()),
                "eliminados_por_score": int(drop_score.sum())})
    keep = alive & ~drop_slices & ~drop_voxels & ~drop_score
    new_ids = {old: new for new, old in enumerate(table.index[keep], start=1)}
    lookup = np.zeros((int(table.index.max()) if len(table) else 0) + 1, dtype=np.uint16)
    for old, new in new_ids.items():
        lookup[old] = new
    for z in range(volume.shape[0]):
        if volume[z].any():
            volume[z] = lookup[volume[z]]
    kept = table.loc[keep].reset_index()
    kept.insert(0, "fragment_id", [new_ids[t] for t in kept["track_id"]])
    return volume, kept, log


## 7. Evaluación por fragmento en 3D

La unidad es el **fragmento 3D** (como en el challenge PENGWIN, *fragmentwise*), en la geometría original del MHA. Cada fragmento predicho se empareja con uno real **de la misma región** por asignación húngara sobre el IoU 3D; se da por acertado con IoU ≥ 0.5 (estricto) o ≥ 0.25 (laxo). Se reporta precisión, recall y F1 por fragmento, Dice/IoU 3D por región, recall por tamaño del fragmento real y la taxonomía de errores (la misma de 23.6b, ahora en 3D):

| Estado (fragmento real) | Significa |
|---|---|
| `ok` | emparejado con IoU ≥ 0.5 |
| `omitido` | menos del 10 % de sus vóxeles fue predicho |
| `clase_incorrecta` | lo cubre otro fragmento, pero de otra región |
| `fusionado` | el fragmento que más lo cubre también cubre ≥ 20 % de otro fragmento real |
| `dividido` | ≥ 2 fragmentos predichos cubren cada uno ≥ 20 % de él |
| `mal_segmentado` | cubierto, pero con IoU < 0.5 sin ser fusión ni división |

Del lado de la predicción: `tp`, `falso_positivo` (< 10 % dentro de la referencia), `parte_de_division`, `fusion`, `clase_incorrecta`, `mal_segmentado`. Los conteos de todos los casos se **suman antes de calcular** F1 y Dice, para que un caso con pocos fragmentos no pese igual que uno con muchos.


In [ ]:
SIZE_BINS_VOX = ((0, 100), (100, 1000), (1000, 10 ** 12))                 # tamaño del fragmento real, en vóxeles
SIZE_BIN_NAMES = ("pequeño(<100 vox)", "mediano(100-1000)", "grande(>=1000)")
MATCH_IOU = (0.5, 0.25)         # umbrales de IoU 3D para dar un fragmento por acertado (0.5 = estricto, 0.25 = laxo)


def class_of_gt_id(fragment_id):
    fragment_id = int(fragment_id)
    if 1 <= fragment_id <= 10:
        return 1
    if 11 <= fragment_id <= 20:
        return 2
    if 21 <= fragment_id <= 30:
        return 3
    raise ValueError(f"ID de fragmento real fuera de 1..30: {fragment_id}")


def contingency(gt_volume, pred_volume):
    """Tabla (id real, id predicho, vóxeles) solo sobre los vóxeles donde hay algo en alguno de los dos volúmenes."""
    if gt_volume.shape != pred_volume.shape:
        raise ValueError(f"El volumen predicho {pred_volume.shape} no tiene la geometría de la etiqueta {gt_volume.shape}")
    union = (gt_volume > 0) | (pred_volume > 0)
    keys = gt_volume[union].astype(np.int64) * 65536 + pred_volume[union].astype(np.int64)
    unique, counts = np.unique(keys, return_counts=True)
    return unique // 65536, unique % 65536, counts


def evaluate_volume_3d(pred_volume, pred_class, gt_volume, thresholds=MATCH_IOU):
    """Evaluación por FRAGMENTO en 3D (la unidad del challenge PENGWIN, 'fragmentwise'), en la geometría original.
    pred_class: {id predicho: clase 1..3}. Empareja por región con asignación húngara sobre el IoU 3D."""
    g_ids, p_ids, counts = contingency(gt_volume, pred_volume)
    gt_size, pred_size, inter = defaultdict(int), defaultdict(int), {}
    gts_of_pred, preds_of_gt = defaultdict(list), defaultdict(list)
    for g, p, n in zip(g_ids.tolist(), p_ids.tolist(), counts.tolist()):
        if g > 0:
            gt_size[g] += n
        if p > 0:
            pred_size[p] += n
        if g > 0 and p > 0:
            inter[(g, p)] = n
            gts_of_pred[p].append(g); preds_of_gt[g].append(p)
    iou = {k: v / (gt_size[k[0]] + pred_size[k[1]] - v) for k, v in inter.items()}

    # --- región (sacro / coxales) en 3D: Dice e IoU acumulados ---
    raw = {}
    for c in (1, 2, 3):
        i_c = sum(n for g, p, n in zip(g_ids.tolist(), p_ids.tolist(), counts.tolist())
                  if g > 0 and p > 0 and class_of_gt_id(g) == c and pred_class.get(p) == c)
        g_c = sum(n for g, n in gt_size.items() if class_of_gt_id(g) == c)
        p_c = sum(n for p, n in pred_size.items() if pred_class.get(p) == c)
        raw[c] = (i_c, g_c, p_c)

    # --- emparejamiento por fragmento ---
    matches = {}
    for thr in thresholds:
        found = []
        for c in (1, 2, 3):
            gts = sorted(g for g in gt_size if class_of_gt_id(g) == c)
            preds = sorted(p for p in pred_size if pred_class.get(p) == c)
            if not gts or not preds:
                continue
            m = np.array([[iou.get((g, p), 0.0) for p in preds] for g in gts])
            for i, j in zip(*linear_sum_assignment(-m)):
                if m[i, j] >= thr:
                    found.append((gts[i], preds[j], float(m[i, j])))
        matches[thr] = found
    strict = matches[thresholds[0]]
    matched_gt, matched_pred = {g: p for g, p, _ in strict}, {p: g for g, p, _ in strict}
    iou_of_match = {g: v for g, _, v in strict}

    # --- taxonomía de errores por fragmento real (misma idea que 23.6b, ahora en 3D) ---
    gt_rows = []
    for gid in sorted(gt_size):
        gs, region = gt_size[gid], class_of_gt_id(gid)
        cover = {p: inter[(gid, p)] / gs for p in preds_of_gt[gid]}
        same = {p: v for p, v in cover.items() if pred_class.get(p) == region}
        other = {p: v for p, v in cover.items() if pred_class.get(p) != region}
        if gid in matched_gt:
            status = "ok"
        elif sum(cover.values()) < 0.10:
            status = "omitido"
        elif sum(same.values()) < 0.10 and sum(other.values()) >= 0.50:
            status = "clase_incorrecta"
        else:
            best = max(same, key=same.get) if same else None
            fused = best is not None and any(
                other_gt != gid and class_of_gt_id(other_gt) == region and inter[(other_gt, best)] / gt_size[other_gt] >= 0.20
                for other_gt in gts_of_pred[best])
            if fused:
                status = "fusionado"
            elif sum(v >= 0.20 for v in same.values()) >= 2:
                status = "dividido"
            else:
                status = "mal_segmentado"
        best_pred = max(same, key=lambda p: iou[(gid, p)]) if same else 0
        gt_rows.append({"gt_id": gid, "region": REGION_OF_CLASS[region], "gt_voxels": gs,
                        "size_bin": next(name for (lo, hi), name in zip(SIZE_BINS_VOX, SIZE_BIN_NAMES) if lo <= gs < hi),
                        "status": status, "matched_pred": matched_gt.get(gid, 0),
                        "iou_3d": iou_of_match.get(gid, iou.get((gid, best_pred), 0.0) if best_pred else 0.0),
                        "n_preds_touching": len(cover), "cover_same_region": round(sum(same.values()), 3)})
    pred_rows = []
    for pid in sorted(pred_size):
        ps, cls = pred_size[pid], pred_class.get(pid)
        inside = sum(inter[(g, pid)] for g in gts_of_pred[pid]) / ps
        if pid in matched_pred:
            status = "tp"
        elif inside < 0.10:
            status = "falso_positivo"
        else:
            g_star = max(gts_of_pred[pid], key=lambda g: inter[(g, pid)])
            if class_of_gt_id(g_star) != cls:
                status = "clase_incorrecta"
            elif inter[(g_star, pid)] / ps >= 0.50 and g_star in matched_gt:
                status = "parte_de_division"       # trozo de un fragmento real que ya está acertado por otro trozo
            elif sum(inter[(g, pid)] / gt_size[g] >= 0.20 for g in gts_of_pred[pid]) >= 2:
                status = "fusion"
            else:
                status = "mal_segmentado"
        pred_rows.append({"pred_id": pid, "region": REGION_OF_CLASS.get(cls, "?"), "pred_voxels": ps,
                          "fraction_inside_gt": round(inside, 3), "status": status,
                          "matched_gt": matched_pred.get(pid, 0)})
    gt_table, pred_table = pd.DataFrame(gt_rows), pd.DataFrame(pred_rows)

    counts_out = {"n_gt": len(gt_size), "n_pred": len(pred_size)}
    for thr in thresholds:
        tp = len(matches[thr])
        counts_out[f"tp@{thr}"], counts_out[f"fp@{thr}"], counts_out[f"fn@{thr}"] = tp, len(pred_size) - tp, len(gt_size) - tp
    for name in SIZE_BIN_NAMES:
        in_bin = gt_table[gt_table["size_bin"] == name] if len(gt_table) else gt_table
        counts_out[f"n_gt[{name}]"] = len(in_bin)
        counts_out[f"tp[{name}]"] = int((in_bin["status"] == "ok").sum()) if len(in_bin) else 0
    for status in ("ok", "omitido", "clase_incorrecta", "fusionado", "dividido", "mal_segmentado"):
        counts_out[f"gt_{status}"] = int((gt_table["status"] == status).sum()) if len(gt_table) else 0
    for status in ("tp", "falso_positivo", "clase_incorrecta", "parte_de_division", "fusion", "mal_segmentado"):
        counts_out[f"pred_{status}"] = int((pred_table["status"] == status).sum()) if len(pred_table) else 0
    for c in (1, 2, 3):
        counts_out[f"inter_{c}"], counts_out[f"gsum_{c}"], counts_out[f"psum_{c}"] = raw[c]
    counts_out["sum_iou_matched"] = float(sum(v for _, _, v in strict))
    return {"counts": counts_out, "gt_table": gt_table, "pred_table": pred_table, "matches": matches}


def summarize_counts(counts, thresholds=MATCH_IOU):
    """Counts (de un caso o sumados de varios) -> métricas. Sumar los conteos y luego dividir evita que un caso con
    pocos fragmentos pese lo mismo que uno con muchos."""
    out = {"n_gt": counts["n_gt"], "n_pred": counts["n_pred"],
           "sobre_segmentacion": counts["n_pred"] / max(counts["n_gt"], 1)}
    for thr in thresholds:
        tp, fp, fn = counts[f"tp@{thr}"], counts[f"fp@{thr}"], counts[f"fn@{thr}"]
        precision, recall = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
        out[f"precision@{thr}"], out[f"recall@{thr}"] = precision, recall
        out[f"f1@{thr}"] = 2 * precision * recall / max(precision + recall, 1e-9)
    dices, ious = [], []
    for c in (1, 2, 3):
        i_c, g_c, p_c = counts[f"inter_{c}"], counts[f"gsum_{c}"], counts[f"psum_{c}"]
        if g_c:
            out[f"dice_{REGION_OF_CLASS[c]}"] = 2 * i_c / max(g_c + p_c, 1)
            out[f"iou_{REGION_OF_CLASS[c]}"] = i_c / max(g_c + p_c - i_c, 1)
            dices.append(out[f"dice_{REGION_OF_CLASS[c]}"]); ious.append(out[f"iou_{REGION_OF_CLASS[c]}"])
    out["dice_regiones"], out["iou_regiones"] = (float(np.mean(dices)) if dices else np.nan), (float(np.mean(ious)) if ious else np.nan)
    out["iou_medio_aciertos"] = counts["sum_iou_matched"] / max(counts["tp@" + str(thresholds[0])], 1)
    for name in SIZE_BIN_NAMES:
        out[f"recall[{name}]"] = counts[f"tp[{name}]"] / counts[f"n_gt[{name}]"] if counts[f"n_gt[{name}]"] else np.nan
    for key, value in counts.items():
        if key.startswith(("gt_", "pred_")):
            out[key] = value
    return out


def sum_counts(list_of_counts):
    total = defaultdict(float)
    for counts in list_of_counts:
        for key, value in counts.items():
            total[key] += value
    return {k: (int(v) if float(v).is_integer() and k != "sum_iou_matched" else v) for k, v in total.items()}


def compare_distances(pred_volume, pred_region_by_id, gt_volume, spacing_xyz, matches_strict, case_id=None):
    """Distancia borde a borde al fragmento principal (EDT con el espaciado del header) en la predicción y en la
    referencia, para los fragmentos reales emparejados. 'main_ok' dice si el principal predicho de su región es
    el principal real: si no, las dos distancias están medidas respecto a huesos distintos y no son comparables."""
    gt_dist = measure_fragment_separations(gt_volume.astype(np.int32), spacing_xyz, case_id=case_id)
    pr_dist = measure_fragment_separations(pred_volume.astype(np.int32), spacing_xyz, case_id=case_id, region_by_id=pred_region_by_id)
    if gt_dist.empty or pr_dist.empty:
        return pd.DataFrame(columns=["gt_id", "pred_id", "region", "iou_3d", "gt_distance_mm", "pred_distance_mm",
                                     "abs_error_mm", "gt_is_main", "main_ok"])
    gt_main = dict(zip(gt_dist["region"], gt_dist["main_fragment_id"]))
    pr_main = dict(zip(pr_dist["region"], pr_dist["main_fragment_id"]))
    gt_by_id, pr_by_id = gt_dist.set_index("fragment_id"), pr_dist.set_index("fragment_id")
    rows = []
    for gid, pid, value in matches_strict:
        region = gt_by_id.at[gid, "region"]
        rows.append({"gt_id": gid, "pred_id": pid, "region": region, "iou_3d": round(value, 3),
                     "gt_distance_mm": gt_by_id.at[gid, "distance_mm"], "pred_distance_mm": pr_by_id.at[pid, "distance_mm"],
                     "abs_error_mm": abs(pr_by_id.at[pid, "distance_mm"] - gt_by_id.at[gid, "distance_mm"]),
                     "gt_is_main": bool(gt_by_id.at[gid, "is_main"]),
                     "main_ok": bool(pr_main.get(region) is not None and gt_main.get(region) is not None
                                     and (gt_main[region], pr_main[region]) in {(g, p) for g, p, _ in matches_strict})})
    return pd.DataFrame(rows)


## 8. Ajuste con `val`, revisión y archivos de salida

`tune_pipeline` ajusta por pasos (un bloque de parámetros a la vez, fijando lo ya elegido), con este criterio: **F1 por fragmento 3D a IoU ≥ 0.5, con los conteos de todos los casos de `val` sumados**. Si varias opciones quedan a menos de 0.003 del mejor F1, gana la que produce menos fragmentos y, si hay una meseta, la del centro. Para ir más rápido el ajuste se hace con el plano submuestreado a la mitad (`STRIDE_TUNE = 2`); la configuración elegida se **vuelve a evaluar a resolución completa**.

> `val` tiene solo 5 casos: el ajuste puede sobreajustarse a ellos. Por eso se guarda todo el historial (`ajuste_completo_val.csv`) y se grafica la sensibilidad al umbral de asociación; si el F1 es plano en un rango, el valor exacto no importa, y si cambia bruscamente hay que desconfiar.


In [ ]:
def assemble_case(bundle, dec_cfg, link_cfg, post_cfg, stride=1, decoded=None, keep_linked=False):
    """Un caso completo: salidas del decoder -> instancias 2D -> geometría MHA -> fragmentos 3D -> limpieza."""
    local, classes, scores = decoded if decoded is not None else decode_and_restore(bundle, dec_cfg, stride)
    linked, table, events = link_instances_3d(local, classes, scores, **link_cfg)
    volume, kept, log = clean_tracks(linked, table, stride=stride, **post_cfg)
    out = {"volume": volume, "table": kept, "events": events, "log": log, "n_tracks_before_cleaning": len(table)}
    if keep_linked:
        out["linked_volume"], out["linked_table"] = linked, table
    return out


class CaseData:
    """Un caso listo para ajustar: salidas del modelo (o del oráculo) + referencia, con la caché de decodificación."""
    def __init__(self, bundle, gt_volume, image_itk=None):
        self.bundle, self.gt, self.image_itk = bundle, gt_volume, image_itk
        self._gt_strided, self._decoded_key, self._decoded = {}, None, None

    def gt_at(self, stride):
        if stride == 1:
            return self.gt
        if stride not in self._gt_strided:
            self._gt_strided[stride] = np.ascontiguousarray(self.gt[:, ::stride, ::stride])
        return self._gt_strided[stride]

    def decoded(self, dec_cfg, stride):
        key = (json.dumps(dec_cfg, sort_keys=True, default=str), stride)
        if key != self._decoded_key:                 # solo se guarda la última (cada una pesa cientos de MB a 512 x 512)
            self._decoded, self._decoded_key = None, None
            gc.collect()
            self._decoded, self._decoded_key = decode_and_restore(self.bundle, dec_cfg, stride), key
        return self._decoded


def evaluate_config(cases, cfg, stride=1):
    """Métricas 3D agregadas (se suman los conteos de todos los casos) para una configuración completa."""
    per_case = []
    for case in cases:
        result = assemble_case(case.bundle, cfg["decode"], cfg["link"], cfg["post"], stride, decoded=case.decoded(cfg["decode"], stride))
        pred_class = dict(zip(result["table"]["fragment_id"], result["table"]["class"]))
        per_case.append(evaluate_volume_3d(result["volume"], pred_class, case.gt_at(stride))["counts"])
        del result
    return summarize_counts(sum_counts(per_case)), per_case


DEFAULT_TUNE_PLAN = [
    # (nombre, bloque de la config, {parámetro: valores a probar}). Cada paso fija el mejor valor y pasa al siguiente.
    ("corte activo", "decode", {"det_gate": [0.05, 0.2, 0.4], "gate_min_run": [1, 3, 5]}),
    ("recorte por cajas", "decode", {"box_margin": [None, 8, 2]}),
    ("picos de centro", "decode", {"center_threshold": [0.15, 0.25, 0.4], "min_instance_pixels": [1, 4, 9]}),
    ("asociación: criterio y umbral", "link", {"overlap_mode": ["min", "iou", "cur"], "min_overlap": [0.05, 0.10, 0.20, 0.35, 0.50]}),
    ("asociación: saltos y trozos", "link", {"max_gap": [0, 1, 2], "many_to_one": [False, True]}),
    ("limpieza: tamaño mínimo", "post", {"min_slices": [1, 2, 3, 5], "min_voxels": [1, 50, 200, 500]}),
    ("limpieza: absorber pequeños", "post", {"absorb_max_voxels": [0, 200, 1000]}),
]
LINK_ONLY_PLAN = [step for step in DEFAULT_TUNE_PLAN if step[1] in ("link", "post")]


def _product(space):
    keys = list(space)
    grids = [[]]
    for k in keys:
        grids = [g + [v] for g in grids for v in space[k]]
    return [dict(zip(keys, g)) for g in grids]


def tune_pipeline(cases, start_cfg, plan=DEFAULT_TUNE_PLAN, stride=2, tolerance=0.003, rounds=1, verbose=True):
    """Ajuste por coordenadas con el conjunto `cases` (val). Criterio: F1 por fragmento 3D a IoU >= 0.5 (suma de todos
    los casos). Si dos opciones quedan a menos de `tolerance` del mejor F1 gana la que produce MENOS fragmentos
    (la más simple); si varias empatan, se toma la del centro de la meseta (no un extremo). Devuelve (config elegida, historial de todos los ensayos)."""
    cfg = json.loads(json.dumps(start_cfg, default=str))
    for block in cfg.values():                       # json convierte None -> null y tuplas -> listas: se restituye
        for key, value in list(block.items()):
            if value == "None":
                block[key] = None
    history, t0 = [], time.time()
    baseline, _ = evaluate_config(cases, cfg, stride)
    history.append({"paso": "línea base", "ronda": 0, "elegido": True, **{f"{b}.{k}": v for b, d in cfg.items() for k, v in d.items()},
                    **{m: baseline[m] for m in ("f1@0.5", "f1@0.25", "precision@0.5", "recall@0.5", "n_pred", "n_gt", "sobre_segmentacion", "dice_regiones")}})
    if verbose:
        print(f"línea base: F1@0.5 = {baseline['f1@0.5']:.3f} | fragmentos predichos {baseline['n_pred']} vs reales {baseline['n_gt']} ({time.time() - t0:.0f} s)")
    for round_index in range(1, rounds + 1):
        for name, block, space in plan:
            trials = []
            for values in _product(space):
                trial = json.loads(json.dumps(cfg, default=str))
                trial[block].update(values)
                metrics, _ = evaluate_config(cases, trial, stride)
                trials.append((values, metrics))
                history.append({"paso": name, "ronda": round_index, "elegido": False,
                                **{f"{b}.{k}": v for b, d in trial.items() for k, v in d.items()},
                                **{m: metrics[m] for m in ("f1@0.5", "f1@0.25", "precision@0.5", "recall@0.5", "n_pred", "n_gt", "sobre_segmentacion", "dice_regiones")}})
            best_f1 = max(m["f1@0.5"] for _, m in trials)
            near = [(v, m) for v, m in trials if m["f1@0.5"] >= best_f1 - tolerance]
            best_key = min((m["n_pred"], round(-m["f1@0.5"], 3)) for _, m in near)       # la opción más simple...
            plateau = [(v, m) for v, m in near if (m["n_pred"], round(-m["f1@0.5"], 3)) == best_key]
            chosen_values, chosen_metrics = plateau[len(plateau) // 2]                   # ...y, si hay meseta, la del centro
            cfg[block].update(chosen_values)
            for row in reversed(history):             # marca el ensayo elegido
                if row["paso"] == name and row["ronda"] == round_index and all(row[f"{block}.{k}"] == v for k, v in chosen_values.items()):
                    row["elegido"] = True
                    break
            if verbose:
                print(f"[ronda {round_index}] {name}: {chosen_values} -> F1@0.5 {chosen_metrics['f1@0.5']:.3f}, "
                      f"fragmentos {chosen_metrics['n_pred']} ({time.time() - t0:.0f} s)")
    return cfg, pd.DataFrame(history)


In [ ]:
def review_events(result, gt_volume):
    """Cruza cada alerta de fusión/división del enlace con la referencia: ¿la alerta era justificada?
    Necesita assemble_case(..., keep_linked=True), porque los eventos nombran fragmentos ANTES de la limpieza."""
    g_ids, p_ids, counts = contingency(gt_volume, result["linked_volume"])
    best, size = {}, defaultdict(int)
    for g, p, n in zip(g_ids.tolist(), p_ids.tolist(), counts.tolist()):
        if p > 0:
            size[p] += n
            if g > 0 and n > best.get(p, (0, 0))[1]:
                best[p] = (g, n)
    dominant = {p: (g if n >= 0.10 * size[p] else 0) for p, (g, n) in best.items()}     # 0 = casi todo fuera de la referencia
    rows = []
    for event in result["events"]:
        tracks = event["tracks"] if event["type"] == "merge_candidate" else event.get("to_tracks", [event["track"]])
        real = {dominant.get(t, 0) for t in tracks} - {0}
        distinct = len(real) > 1
        if event["type"] == "merge_candidate":
            verdict = "una instancia cubre fragmentos reales DISTINTOS (fusión del decoder)" if distinct else "pistas del mismo fragmento real (duplicado)"
        elif len(set(tracks)) == 1:
            verdict = "trozos unidos en un mismo fragmento 3D (many_to_one): resuelta"
        else:
            verdict = "trozos de fragmentos reales distintos (división legítima)" if distinct else "trozos del MISMO fragmento real (sobre-segmentación)"
        rows.append({"tipo": event["type"], "corte": event["z"], "region": REGION_OF_CLASS[event["class"]],
                     "fragmentos_involucrados": tracks, "fragmentos_reales": sorted(real), "veredicto": verdict})
    return pd.DataFrame(rows)


def gt_z_ranges(gt_volume):
    present = np.flatnonzero((gt_volume > 0).any(axis=(1, 2)))
    ranges = {}
    for z in present:
        for i in np.unique(gt_volume[z]):
            if i > 0:
                lo, hi = ranges.get(int(i), (z, z))
                ranges[int(i)] = (min(lo, z), max(hi, z))
    return ranges


def plot_track_timeline(table, gt_volume=None, title="", ax=None):
    """Cada fila = un fragmento 3D; la barra abarca los cortes en los que aparece. Los fragmentos muy cortos o pequeños
    (sospechosos de ser ruido) llevan borde rojo. Con la referencia, se dibujan debajo los fragmentos reales."""
    own = ax is None
    if own:
        fig, ax = plt.subplots(figsize=(11, 0.18 * max(len(table), 6) + 2.2))
    rows = table.sort_values(["class", "z_min"]).reset_index(drop=True)
    for y, row in rows.iterrows():
        suspicious = row["n_slices"] <= 2 or row["n_voxels"] < 200
        ax.barh(y, row["z_max"] - row["z_min"] + 1, left=row["z_min"], height=0.8, color=TARGET_COLORS[int(row["class"])],
                edgecolor="red" if suspicious else "none", lw=1.2, alpha=0.9)
    offset = len(rows) + 1
    if gt_volume is not None:
        for k, (gid, (lo, hi)) in enumerate(sorted(gt_z_ranges(gt_volume).items(), key=lambda kv: (class_of_gt_id(kv[0]), kv[1][0]))):
            ax.barh(offset + k, hi - lo + 1, left=lo, height=0.8, color=TARGET_COLORS[class_of_gt_id(gid)], hatch="//", alpha=0.55)
        ax.axhline(offset - 1, color="gray", lw=0.8)
        ax.text(0.99, offset - 0.9, "referencia (hachurado) ↓   predicción ↑", transform=ax.get_yaxis_transform(), fontsize=8, color="gray", ha="right", va="top")
    ax.set_xlabel("corte (z)"); ax.set_yticks([]); ax.set_title(title or "Fragmentos 3D por corte (borde rojo = ≤2 cortes o <200 vóxeles)")
    if own:
        plt.tight_layout()
    return ax


def _instance_colors(volume, table_class):
    lut = np.zeros((int(volume.max()) + 1 if volume.size else 1, 3))
    for fid, cls in table_class.items():
        base = np.array(plt.matplotlib.colors.to_rgb(TARGET_COLORS[int(cls)]))
        shade = [0.0, 0.35, 0.6, 0.2, 0.5][fid % 5]
        lut[fid] = base * (1 - shade) + shade
    return lut


def plot_mip_comparison(gt_volume, pred_volume, pred_class, spacing_xyz, title=""):
    """Proyección coronal (máximo sobre y) de referencia vs predicción, con la proporción física correcta.
    Es la vista rápida para ver fusiones, divisiones y fragmentos sueltos de todo el volumen a la vez."""
    gt_class = {int(i): class_of_gt_id(i) for i in np.unique(gt_volume) if i > 0}
    fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
    for ax, volume, classes, name in ((axes[0], gt_volume, gt_class, "referencia"), (axes[1], pred_volume, pred_class, "predicción")):
        proj = volume.max(axis=1)
        image = _instance_colors(volume, classes)[proj] * (proj > 0)[..., None]
        ax.imshow(image, origin="lower", aspect=spacing_xyz[2] / spacing_xyz[0], interpolation="nearest")
        ax.set_title(f"{name}: {len(classes)} fragmentos"); ax.set_xlabel("x"); ax.set_ylabel("z")
    fig.suptitle(title); fig.tight_layout()
    return fig


def small_fragment_report(gt_table):
    """Recall por tamaño del fragmento real y lista de los pequeños que no se acertaron."""
    by_size = gt_table.groupby("size_bin")["status"].agg(n="size", acertados=lambda s: int((s == "ok").sum()))
    by_size["recall"] = (by_size["acertados"] / by_size["n"]).round(3)
    missed = gt_table[(gt_table["status"] != "ok") & (gt_table["size_bin"] == SIZE_BIN_NAMES[0])]
    return by_size, missed


def save_volume_mha(path, volume_zyx, reference_itk):
    """Guarda el volumen de instancias con el MISMO header que el MHA original (tamaño, spacing, origen, dirección)."""
    image = sitk.GetImageFromArray(volume_zyx.astype(np.uint16))
    image.CopyInformation(reference_itk)
    sitk.WriteImage(image, str(path), useCompression=True)


def save_config_3d(cfg, path, extra=None):
    payload = {"config": cfg, "creado": time.strftime("%Y-%m-%d %H:%M"), **(extra or {})}
    Path(path).write_text(json.dumps(payload, indent=2, ensure_ascii=False, default=str), encoding="utf-8")


def load_config_3d(path):
    payload = json.loads(Path(path).read_text(encoding="utf-8"))
    return payload["config"], payload


def predict_case_3d(comp, case_id, cfg, cfg_slice=CFG_SLICE, bundle=None, verbose=True):
    """Pipeline completo de UN volumen, sin usar la etiqueta: todos los cortes -> detección + NMS -> segmentación ->
    geometría original -> fragmentos 3D. El dict tiene las mismas claves que inferir_volumen (Fabián) que usa
    measure_fragment_separations: instance_volume, region_by_id, spacing_xyz, fragmentos, eventos."""
    volume, _, image_itk = load_case_volumes(case_id, with_label=False)
    if bundle is None:
        bundle = infer_case(comp, case_id, cfg_slice, volume=volume, image_itk=image_itk, verbose=verbose)
    result = assemble_case(bundle, cfg["decode"], cfg["link"], cfg["post"], stride=1, keep_linked=True)
    table = result["table"]
    return {"case_id": case_id, "instance_volume": result["volume"], "spacing_xyz": tuple(image_itk.GetSpacing()),
            "region_by_id": {int(f): REGION_OF_CLASS[int(c)] for f, c in zip(table["fragment_id"], table["class"])},
            "fragmentos": table, "eventos": result["events"], "log_limpieza": result["log"], "config": cfg,
            "image_itk": image_itk, "_assembled": result}


In [ ]:
def save_bundle(bundle, folder, tag):
    """Guarda las salidas crudas de un caso (para no repetir la red al cambiar umbrales). `tag` identifica el modelo."""
    folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)
    stem = folder / f"bundle_{bundle['case_id']}"
    np.savez(stem.with_suffix(".npz"), sem=bundle["sem"], ctr=bundle["ctr"], off=bundle["off"], has_seg=bundle["has_seg"])
    meta = {"case_id": bundle["case_id"], "source": bundle["source"], "shape": list(bundle["shape"]),
            "spacing_xyz": list(bundle["spacing_xyz"]), "transform": bundle["transform"], "tag": tag,
            "dets": [None if d is None else {k: v.tolist() for k, v in d.items()} for d in bundle["dets"]]}
    stem.with_suffix(".json").write_text(json.dumps(meta), encoding="utf-8")


def load_bundle(folder, case_id, tag):
    stem = Path(folder) / f"bundle_{case_id}"
    if not (stem.with_suffix(".npz").is_file() and stem.with_suffix(".json").is_file()):
        return None
    meta = json.loads(stem.with_suffix(".json").read_text(encoding="utf-8"))
    if meta["tag"] != tag:
        return None                                  # otro modelo o otra configuración de inferencia: no se reutiliza
    arrays = np.load(stem.with_suffix(".npz"))
    dets = [None if d is None else {"boxes": np.asarray(d["boxes"], np.float32).reshape(-1, 4),
                                    "scores": np.asarray(d["scores"], np.float32), "labels": np.asarray(d["labels"], np.int64)}
            for d in meta["dets"]]
    return {"case_id": meta["case_id"], "source": meta["source"], "shape": tuple(meta["shape"]),
            "spacing_xyz": tuple(meta["spacing_xyz"]), "transform": meta["transform"], "sem": arrays["sem"],
            "ctr": arrays["ctr"], "off": arrays["off"], "has_seg": arrays["has_seg"], "dets": dets}


def per_case_table(cases, per_case_counts):
    rows = []
    for case, counts in zip(cases, per_case_counts):
        rows.append({"caso": case.bundle["case_id"], **summarize_counts(counts)})
    rows.append({"caso": "TODOS", **summarize_counts(sum_counts(per_case_counts))})
    return pd.DataFrame(rows)


SUMMARY_COLUMNS = ["caso", "n_gt", "n_pred", "sobre_segmentacion", "precision@0.5", "recall@0.5", "f1@0.5", "f1@0.25",
                   "dice_regiones", "iou_regiones", f"recall[{SIZE_BIN_NAMES[0]}]", f"recall[{SIZE_BIN_NAMES[1]}]", f"recall[{SIZE_BIN_NAMES[2]}]"]
ERROR_COLUMNS = ["caso", "gt_ok", "gt_omitido", "gt_clase_incorrecta", "gt_fusionado", "gt_dividido", "gt_mal_segmentado",
                 "pred_tp", "pred_falso_positivo", "pred_parte_de_division", "pred_fusion", "pred_clase_incorrecta", "pred_mal_segmentado"]


def plot_link_curves(history):
    """F1 por fragmento 3D en función del umbral de asociación, una curva por criterio de solapamiento."""
    rows = history[history["paso"] == "asociación: criterio y umbral"]
    if rows.empty:
        return None
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6), constrained_layout=True)
    for mode, group in rows.groupby("link.overlap_mode"):
        group = group.sort_values("link.min_overlap")
        axes[0].plot(group["link.min_overlap"], group["f1@0.5"], marker="o", label=mode)
        axes[1].plot(group["link.min_overlap"], group["n_pred"], marker="o", label=mode)
    axes[1].axhline(rows["n_gt"].iloc[0], color="gray", ls="--", lw=1, label="fragmentos reales")
    axes[0].set_title("F1 por fragmento (IoU 3D ≥ 0.5)"); axes[1].set_title("Fragmentos 3D predichos")
    for ax in axes:
        ax.set_xlabel("umbral de asociación (min_overlap)"); ax.legend(title="criterio")
    return fig


def plot_slice_overlay(volume_hu, gt_volume, pred_volume, pred_class, z, title=""):
    """Un corte con la referencia (contornos blancos) y la predicción (color por región)."""
    image = enhance_bone_contrast(apply_hu_window(volume_hu[z]))
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    for ax, vol, name in ((axes[0], gt_volume, "referencia"), (axes[1], pred_volume, "predicción")):
        ax.imshow(image, cmap="gray")
        for fid in np.unique(vol[z]):
            if fid == 0:
                continue
            cls = class_of_gt_id(fid) if name == "referencia" else pred_class.get(int(fid))
            ax.contour(vol[z] == fid, levels=[0.5], colors=[TARGET_COLORS[int(cls)] if cls else "yellow"], linewidths=1.0)
        ax.set_title(f"{name} · corte {z}"); ax.axis("off")
    fig.suptitle(title)
    return fig


## 9. Pruebas con errores provocados a propósito

Volúmenes sintéticos diminutos (no usan imágenes médicas ni el modelo). Cada pieza debe reaccionar al error que se le provoca; si alguna aserción falla, el módulo correspondiente tiene un defecto.


In [ ]:
# ---- pruebas con volúmenes sintéticos diminutos: cada pieza debe reaccionar a un error provocado a propósito ----
def _box(volume, fragment_id, z, y, x):
    volume[z[0]:z[1], y[0]:y[1], x[0]:x[1]] = fragment_id


def _local_stack(slices):
    """Lista de (máscara_local, {id: clase}) -> entradas de link_instances_3d."""
    local = np.stack([m for m, _ in slices]).astype(np.uint16)
    return local, [c for _, c in slices], [{k: 0.9 for k in c} for _, c in slices]


# 1) Mesetas: dos píxeles vecinos con el mismo valor son UN pico (la versión de 23.x contaba dos)
heat = np.full((30, 30), -5.0, np.float32); heat[10, 10] = heat[10, 11] = 3.0; heat[25, 25] = 2.0
assert len(_pick_peaks(heat, _logit(0.25), 7)) == 2, "meseta + pico aparte = 2 picos"
assert int(((heat == ndimage.maximum_filter(heat, size=15)) & (heat >= _logit(0.25))).sum()) == 3, "sin la corrección serían 3"

# 2) Asociación: un fragmento pequeño que aparece DENTRO de la sección del grande no debe robarle la identidad
big = np.zeros((40, 40), np.uint16); big[5:35, 5:35] = 1
shifted = np.zeros((40, 40), np.uint16); shifted[5:35, 6:36] = 1           # el grande se corre 1 píxel entre cortes
ring = shifted.copy(); ring[15:20, 15:20] = 0
small = np.zeros((40, 40), np.uint16); small[15:20, 15:20] = 2
stack = [(big, {1: 1}), (big, {1: 1}), (ring + small, {1: 1, 2: 1}), (ring + small, {1: 1, 2: 1})]
for mode, keeps_identity in (("min", False), ("iou", True)):
    vol_, table_, _ = link_instances_3d(*_local_stack(stack), min_overlap=0.10, overlap_mode=mode)
    same_big = vol_[0, 10, 10] == vol_[3, 10, 10]
    assert bool(same_big) == keeps_identity, f"criterio '{mode}': identidad del fragmento grande {same_big}"
    assert len(table_) == 2
print("Asociación: con 'min' (23.10) el pequeño le roba la identidad al grande; con 'iou' no. OK")

# 3) Saltos: el fragmento falta en un corte
blob = np.zeros((40, 40), np.uint16); blob[10:20, 10:20] = 1
gap_stack = [(blob, {1: 2}), (blob, {1: 2}), (np.zeros_like(blob), {}), (blob, {1: 2}), (blob, {1: 2})]
assert len(link_instances_3d(*_local_stack(gap_stack), max_gap=0)[1]) == 2
assert len(link_instances_3d(*_local_stack(gap_stack), max_gap=1)[1]) == 1
print("Saltos: max_gap=0 parte el fragmento en 2; max_gap=1 lo une. OK")

# 4) Una sección que sale en dos manchas
two = np.zeros((40, 40), np.uint16); two[10:20, 5:15] = 1; two[10:20, 25:35] = 2
one = np.zeros((40, 40), np.uint16); one[10:20, 5:35] = 1
split_stack = [(one, {1: 3}), (one, {1: 3}), (two, {1: 3, 2: 3}), (two, {1: 3, 2: 3})]
vol_a, table_a, events_a = link_instances_3d(*_local_stack(split_stack), min_overlap=0.10, overlap_mode="iou")
vol_b, table_b, _ = link_instances_3d(*_local_stack(split_stack), min_overlap=0.10, overlap_mode="min", many_to_one=True)
assert len(table_b) == 1 and any(e["type"] == "split_candidate" for e in link_instances_3d(*_local_stack(split_stack), overlap_mode="min")[2])
print("Trozos: many_to_one=True mantiene un solo fragmento y la división queda registrada como alerta. OK")

# 5) Taxonomía de errores 3D: cada contador debe reaccionar
gt_v = np.zeros((10, 60, 240), np.int16)
_box(gt_v, 1, (0, 10), (0, 10), (0, 10))        # sacro grande: acertado
_box(gt_v, 2, (0, 5), (20, 25), (0, 5))         # sacro pequeño: omitido
_box(gt_v, 3, (0, 10), (30, 40), (0, 10))       # sacro: mal segmentado
for k, fid in enumerate((11, 12, 13)):          # tres coxales izquierdos pegados que el modelo funde en uno
    _box(gt_v, fid, (0, 10), (0, 10), (20 + 10 * k, 30 + 10 * k))
_box(gt_v, 21, (0, 9), (20, 29), (100, 109))    # coxal derecho partido en tres trozos
_box(gt_v, 22, (0, 10), (40, 50), (100, 110))   # coxal derecho que el modelo etiqueta como izquierdo
pred_v = np.zeros_like(gt_v, dtype=np.uint16)
_box(pred_v, 1, (0, 10), (0, 10), (0, 10))
_box(pred_v, 2, (0, 10), (30, 40), (0, 3))      # cubre ~30 % de GT 3 y nada más: mal segmentado
_box(pred_v, 2, (0, 10), (30, 40), (10, 20))    # ... + píxeles fuera de la referencia
_box(pred_v, 3, (0, 10), (0, 10), (20, 50))     # funde 11, 12 y 13
for k in range(3):
    _box(pred_v, 4 + k, (0, 9), (20, 29), (100 + 3 * k, 103 + 3 * k))
_box(pred_v, 7, (0, 10), (40, 50), (100, 110))  # misma caja que GT 22, pero clase 2
_box(pred_v, 8, (0, 10), (50, 60), (200, 210))  # falso positivo lejos de todo
pred_cls = {1: 1, 2: 1, 3: 2, 4: 3, 5: 3, 6: 3, 7: 2, 8: 3}
ev_ = evaluate_volume_3d(pred_v, pred_cls, gt_v)
status_gt = dict(zip(ev_["gt_table"]["gt_id"], ev_["gt_table"]["status"]))
assert status_gt == {1: "ok", 2: "omitido", 3: "mal_segmentado", 11: "fusionado", 12: "fusionado", 13: "fusionado",
                     21: "dividido", 22: "clase_incorrecta"}, status_gt
status_pred = dict(zip(ev_["pred_table"]["pred_id"], ev_["pred_table"]["status"]))
assert status_pred[1] == "tp" and status_pred[3] == "fusion" and status_pred[8] == "falso_positivo" and status_pred[7] == "clase_incorrecta"
summary_ = summarize_counts(ev_["counts"])
assert ev_["counts"]["tp@0.5"] == 1 and ev_["counts"]["fn@0.5"] == 7 and 0 < summary_["f1@0.5"] < 0.2
print("Evaluación 3D: omitido, mal_segmentado, fusionado, dividido, clase_incorrecta y falso_positivo reaccionan. OK")
try:
    contingency(gt_v, pred_v[:, :, :100]); raise AssertionError("debía rechazar geometrías distintas")
except ValueError:
    print("Evaluación 3D: rechaza un volumen predicho con otra geometría que la etiqueta. OK")

# 6) Limpieza: absorber conserva los vóxeles; el filtro de cortes elimina el ruido de un solo corte
vol_c = np.zeros((6, 30, 30), np.uint16); vol_c[:, 5:25, 5:25] = 1; vol_c[2:4, 5:9, 27:30] = 2; vol_c[0, 28, 28] = 3
table_c = pd.DataFrame({"track_id": [1, 2, 3], "class": [1, 1, 1], "n_voxels": [int((vol_c == i).sum()) for i in (1, 2, 3)],
                        "n_slices": [6, 2, 1], "z_min": [0, 2, 0], "z_max": [5, 3, 0], "mean_score": [0.9, 0.9, 0.9]})
cleaned, kept, log_ = clean_tracks(vol_c, table_c, min_slices=2, absorb_max_voxels=200)
assert log_["absorbidos"] == 0 and log_["eliminados_por_cortes"] == 1 and len(kept) == 2, (log_, len(kept))
vol_c[2:4, 5:9, 25:27] = 2                      # ahora el pequeño TOCA al grande
table_c["n_voxels"] = [int((vol_c == i).sum()) for i in (1, 2, 3)]
before = int((vol_c > 0).sum()) - 1               # sin el vóxel suelto del corte 0
cleaned, kept, log_ = clean_tracks(vol_c, table_c, min_slices=2, absorb_max_voxels=200)
assert log_["absorbidos"] == 1 and len(kept) == 1 and int((cleaned > 0).sum()) == before, (log_, int((cleaned > 0).sum()), before)
print("Limpieza: el filtro de cortes elimina el ruido; absorber une el trozo pegado sin perder ni un vóxel. OK")

# 7) Techo de todo el tramo (letterbox -> decoder -> restauración -> asociación -> evaluación) con una etiqueta de forma no cuadrada
gt_small = np.zeros((12, 60, 100), np.int16)
_box(gt_small, 1, (1, 11), (10, 40), (5, 40)); _box(gt_small, 2, (4, 8), (45, 50), (60, 70)); _box(gt_small, 11, (1, 11), (10, 50), (55, 95))
oracle_ = oracle_bundle("sintetico", gt_small, None)
out_ = assemble_case(oracle_, dict(CFG_DECODE), {**CFG_LINK, "overlap_mode": "iou"}, dict(CFG_POST))
ev_o = evaluate_volume_3d(out_["volume"], dict(zip(out_["table"]["fragment_id"], out_["table"]["class"])), gt_small)
assert summarize_counts(ev_o["counts"])["f1@0.5"] == 1.0 and out_["volume"].shape == gt_small.shape
print("Techo con máscaras reales: forma restaurada", out_["volume"].shape, "| F1 por fragmento = 1.0. OK")
print("Todas las pruebas pasaron.")


## 10. Ejecución sobre los casos de `val`


### 10.1 Techo del pipeline con máscaras reales

Salidas del decoder fabricadas desde la etiqueta real → mismas funciones de decodificación, restauración, asociación y evaluación. **Esto es el máximo que podría lograr el modelo con el pipeline actual.** Lo que falle aquí no es culpa de la red: es resolución (los fragmentos muy pequeños pueden perderse al pasar de 512 a 256 píxeles), geometría o criterio de asociación. Se usa la configuración de 23.10 (`min` con 0.10).


In [ ]:
STRIDE_TUNE = 2                       # ajuste en la mitad de resolución en el plano (4x más rápido); la evaluación final es a resolución completa
BASE_CFG = {"decode": dict(CFG_DECODE), "link": dict(CFG_LINK), "post": dict(CFG_POST)}

val_oracle = []
for case_id in CASES["val"]:
    _, label, header = load_case_volumes(case_id, with_label=True, with_image=False)
    val_oracle.append(CaseData(oracle_bundle(case_id, label, header), label, header))
    print(f"caso {case_id}: {label.shape} | spacing {tuple(round(s, 2) for s in header.GetSpacing())} mm | "
          f"{len(np.unique(label)) - 1} fragmentos reales | {int((label > 0).any(axis=(1, 2)).sum())} cortes con hueso")

_, per_case_oracle = evaluate_config(val_oracle, BASE_CFG, stride=1)
table_oracle = per_case_table(val_oracle, per_case_oracle)
display(table_oracle[SUMMARY_COLUMNS].round(3))
display(table_oracle[ERROR_COLUMNS])


### 10.2 Umbral de asociación con máscaras reales de `val`

Mismo ensayo, pero buscando el mejor criterio/umbral de asociación **sin depender del modelo**. Se prueban los tres criterios de solapamiento y cinco umbrales (y luego saltos, trozos y limpieza). La gráfica muestra cuánto importa el umbral: una curva plana significa que el valor exacto no es crítico; una caída fuerte marca el rango que hay que evitar.


In [ ]:
cfg_link_oracle, hist_oracle = tune_pipeline(val_oracle, BASE_CFG, plan=LINK_ONLY_PLAN, stride=STRIDE_TUNE)
hist_oracle.to_csv(OUT_3D_DIR / "ajuste_asociacion_mascaras_reales_val.csv", index=False)
plot_link_curves(hist_oracle); plt.savefig(OUT_3D_DIR / "curvas_asociacion_mascaras_reales.png", dpi=150); plt.show()
_, per_case_oracle2 = evaluate_config(val_oracle, cfg_link_oracle, stride=1)
compare = pd.concat([per_case_table(val_oracle, per_case_oracle).iloc[[-1]].assign(configuracion="23.10 (línea base)"),
                     per_case_table(val_oracle, per_case_oracle2).iloc[[-1]].assign(configuracion="ajustada con máscaras reales")])
display(compare[["configuracion"] + SUMMARY_COLUMNS[1:]].round(3))
print("Asociación elegida con máscaras reales:", cfg_link_oracle["link"], "| limpieza:", cfg_link_oracle["post"])


### 10.3 Inferencia del modelo sobre todos los cortes de `val`

Se procesan **todos los cortes de cada volumen**, incluidos los que no tienen hueso. Las salidas crudas se guardan en `evidencias/salida_3d/cache/` (unos 300 MB por caso; agregar esa carpeta a `.gitignore`). Si el modelo o `CFG_SLICE` cambian, la caché se invalida sola.


In [ ]:
# Libera la memoria de 10.1/10.2 (cada caso con máscaras reales ocupa cientos de MB); para repetirlas, vuelve a ejecutar esas celdas.
for _name in ("val_oracle", "per_case_oracle", "per_case_oracle2"):
    globals().pop(_name, None)
gc.collect()

comp, comp_origin = get_components()
HAVE_MODEL = comp is not None
print("Modelo:", comp_origin)

CACHE_DIR = OUT_3D_DIR / "cache"            # salidas crudas por caso (~300 MB cada una): NO subir a GitHub
CACHE_TAG = f"{comp_origin}|{json.dumps(CFG_SLICE, sort_keys=True, default=str)}"
USE_CACHE = True

val_model = []
if HAVE_MODEL:
    for case_id in CASES["val"]:
        volume, label, header = load_case_volumes(case_id, with_label=True, with_image=True)
        bundle = load_bundle(CACHE_DIR, case_id, CACHE_TAG) if USE_CACHE else None
        if bundle is None:
            t0 = time.time()
            bundle = infer_case(comp, case_id, CFG_SLICE, volume=volume, image_itk=header)
            print(f"caso {case_id}: {volume.shape[0]} cortes en {time.time() - t0:.0f} s | "
                  f"cortes con alguna caja: {int(bundle['has_seg'].sum())}")
            if USE_CACHE:
                save_bundle(bundle, CACHE_DIR, CACHE_TAG)
        else:
            print(f"caso {case_id}: salidas cargadas de la caché")
        val_model.append(CaseData(bundle, label, header))
        del volume
        gc.collect()
else:
    print("Sin pesos entrenados: las celdas desde 10.3 se saltan. Lo anterior (techo y asociación con máscaras reales) sí es válido.")


### 10.4 Línea base con el modelo (parámetros de 23.10 / sección 24)


In [ ]:
if HAVE_MODEL:
    _, per_case_base = evaluate_config(val_model, BASE_CFG, stride=1)
    table_base = per_case_table(val_model, per_case_base)
    table_base.to_csv(OUT_3D_DIR / "linea_base_val.csv", index=False)
    display(table_base[SUMMARY_COLUMNS].round(3))
    display(table_base[ERROR_COLUMNS])


### 10.5 Ajuste con `val`

Recorre: cortes activos → recorte por cajas → picos de centro → criterio y umbral de asociación → saltos y trozos → tamaño mínimo → absorción (puede tardar decenas de minutos en CPU: cada ensayo recorre todos los cortes de los 5 casos). La tabla compara la línea base con la configuración ajustada (ambas a resolución completa) y se guarda `config_3d.json`, que es lo que usarán el dashboard y la evaluación final.


In [ ]:
if HAVE_MODEL:
    best_cfg, hist = tune_pipeline(val_model, BASE_CFG, plan=DEFAULT_TUNE_PLAN, stride=STRIDE_TUNE, rounds=1)
    hist.to_csv(OUT_3D_DIR / "ajuste_completo_val.csv", index=False)
    plot_link_curves(hist); plt.savefig(OUT_3D_DIR / "curvas_asociacion_val.png", dpi=150); plt.show()

    _, per_case_best = evaluate_config(val_model, best_cfg, stride=1)          # evaluación final a resolución completa
    table_best = per_case_table(val_model, per_case_best)
    table_best.to_csv(OUT_3D_DIR / "resultado_val_config_ajustada.csv", index=False)
    comparison = pd.concat([table_base.iloc[[-1]].assign(configuracion="línea base (23.10)"),
                            table_best.iloc[[-1]].assign(configuracion="ajustada con val")])
    display(comparison[["configuracion"] + SUMMARY_COLUMNS[1:]].round(3))
    display(table_best[["caso"] + ERROR_COLUMNS[1:]])
    save_config_3d(best_cfg, OUT_3D_DIR / "config_3d.json",
                   extra={"modelo": comp_origin, "casos_de_ajuste": CASES["val"], "stride_de_ajuste": STRIDE_TUNE,
                          "criterio": "F1 por fragmento 3D, IoU>=0.5, conteos sumados de todos los casos de val"})
    print("Configuración guardada en", OUT_3D_DIR / "config_3d.json")
    print(json.dumps(best_cfg, indent=1, default=str))


### 10.6 Revisión de un caso: fusiones, divisiones y fragmentos pequeños

1. **Alertas del enlace contrastadas con la referencia.** Cada `merge_candidate` o `split_candidate` se clasifica como *justificado* (involucra fragmentos reales distintos) o *redundante* (trozos del mismo fragmento real). Las primeras son las peligrosas: indican que el decoder fundió o separó fragmentos reales.
2. **Recall por tamaño del fragmento real** y lista de los pequeños no acertados.
3. **Fragmentos predichos que no son `tp`** y fragmentos reales con error.
4. **Línea de tiempo** (en qué cortes vive cada fragmento; borde rojo = ≤ 2 cortes o < 200 vóxeles) y **proyección coronal** de referencia vs predicción.


In [ ]:
CASE_REVIEW = CASES["val"][0]                       # cambiar para revisar otro caso
if HAVE_MODEL:
    case = next(c for c in val_model if c.bundle["case_id"] == CASE_REVIEW)
    result = assemble_case(case.bundle, best_cfg["decode"], best_cfg["link"], best_cfg["post"], stride=1, keep_linked=True)
    pred_class = dict(zip(result["table"]["fragment_id"], result["table"]["class"]))
    evaluation = evaluate_volume_3d(result["volume"], pred_class, case.gt)

    # 1) Alertas de fusión/división del enlace, contrastadas con la referencia
    events_df = review_events(result, case.gt)
    if len(events_df):
        display(events_df.groupby(["tipo", "veredicto"]).size().rename("alertas").reset_index())
        display(events_df.head(10))
    else:
        print("El enlace no generó alertas de fusión ni división.")

    # 2) Fragmentos pequeños: ¿se pierden?
    by_size, missed_small = small_fragment_report(evaluation["gt_table"])
    display(by_size)
    print("Fragmentos pequeños no acertados:"); display(missed_small)

    # 3) Los fragmentos predichos que no corresponden a nada real, y los reales que se fundieron/dividieron
    display(evaluation["pred_table"][evaluation["pred_table"]["status"] != "tp"].sort_values("pred_voxels", ascending=False).head(15))
    display(evaluation["gt_table"][evaluation["gt_table"]["status"] != "ok"])

    # 4) Vistas: cortes en los que vive cada fragmento y proyección coronal
    plot_track_timeline(result["table"], case.gt, title=f"Caso {CASE_REVIEW}: fragmentos 3D vs referencia")
    plt.savefig(OUT_3D_DIR / f"linea_de_tiempo_{CASE_REVIEW}.png", dpi=150); plt.show()
    plot_mip_comparison(case.gt, result["volume"], pred_class, case.bundle["spacing_xyz"], f"Caso {CASE_REVIEW}")
    plt.savefig(OUT_3D_DIR / f"mip_coronal_{CASE_REVIEW}.png", dpi=150); plt.show()


### 10.7 Distancia de separación: predicción vs referencia

Es la comparación que pide la sección 3.3 de la consigna: la distancia borde a borde (EDT con el espaciado del header) de cada fragmento al principal de su región, medida en la máscara predicha y en la de referencia. Solo se comparan los fragmentos reales **acertados** y solo cuando el principal predicho de su región es el principal real (`main_ok`); si no, las dos distancias están medidas respecto a huesos distintos. El error de la distancia incluye el error de segmentación **y** el de asociación 3D.


In [ ]:
if HAVE_MODEL:
    distance_tables = []
    for case in val_model:
        result_c = assemble_case(case.bundle, best_cfg["decode"], best_cfg["link"], best_cfg["post"], stride=1)
        table_c = result_c["table"]
        pred_class_c = dict(zip(table_c["fragment_id"], table_c["class"]))
        eval_c = evaluate_volume_3d(result_c["volume"], pred_class_c, case.gt)
        region_by_id_c = {int(f): REGION_OF_CLASS[int(c)] for f, c in zip(table_c["fragment_id"], table_c["class"])}
        distance_tables.append(compare_distances(result_c["volume"], region_by_id_c, case.gt, case.bundle["spacing_xyz"],
                                                 eval_c["matches"][MATCH_IOU[0]], case.bundle["case_id"]).assign(caso=case.bundle["case_id"]))
        del result_c
    distances_val = pd.concat(distance_tables, ignore_index=True)
    distances_val.to_csv(OUT_3D_DIR / "distancias_pred_vs_gt_val.csv", index=False)
    comparable = distances_val[~distances_val["gt_is_main"] & distances_val["main_ok"]]
    print(f"Fragmentos reales acertados (IoU 3D ≥ 0.5): {len(distances_val)} | conminutos comparables (principal bien identificado): {len(comparable)}")
    if len(comparable):
        print(f"Error absoluto de la distancia: media {comparable['abs_error_mm'].mean():.2f} mm | "
              f"mediana {comparable['abs_error_mm'].median():.2f} mm | máx {comparable['abs_error_mm'].max():.2f} mm")
    display(distances_val.round(2))


### 10.8 Archivos de salida

Por caso: el volumen de instancias como **MHA con el header original** (tamaño, spacing, origen y dirección idénticos a la tomografía), la tabla de fragmentos y `region_by_id`. `predict_case_3d` devuelve un diccionario con las mismas claves que `inferir_volumen` (`instance_volume`, `region_by_id`, `spacing_xyz`, `fragmentos`, `eventos`), así que es el punto de entrada para `measure_fragment_separations` y para los visualizadores 2 y 3.


In [ ]:
if HAVE_MODEL:
    for case in val_model:
        out_case = predict_case_3d(comp, case.bundle["case_id"], best_cfg, bundle=case.bundle, verbose=False)
        stem = OUT_3D_DIR / f"pred_{case.bundle['case_id']}"
        save_volume_mha(stem.with_suffix(".mha"), out_case["instance_volume"], out_case["image_itk"])
        out_case["fragmentos"].assign(region=lambda d: d["class"].map(REGION_OF_CLASS)).to_csv(stem.with_name(stem.name + "_fragmentos.csv"), index=False)
        (stem.with_name(stem.name + "_region_by_id.json")).write_text(json.dumps(out_case["region_by_id"]), encoding="utf-8")
        print(f"caso {case.bundle['case_id']}: {len(out_case['fragmentos'])} fragmentos 3D -> {stem.with_suffix('.mha').name} (mismo header que el MHA original)")


### 10.9 Test (una sola vez, con todo congelado)


In [ ]:
EVALUAR_TEST = False        # True SOLO cuando la configuración y el modelo estén congelados: test se evalúa UNA vez
if EVALUAR_TEST and HAVE_MODEL:
    frozen_cfg, frozen_meta = load_config_3d(OUT_3D_DIR / "config_3d.json")
    test_cases, test_counts = [], []
    for case_id in CASES["test"]:
        volume, label, header = load_case_volumes(case_id, with_label=True, with_image=True)
        bundle = infer_case(comp, case_id, CFG_SLICE, volume=volume, image_itk=header, verbose=False)
        test_cases.append(CaseData(bundle, label, header)); del volume
        _, counts = evaluate_config([test_cases[-1]], frozen_cfg, stride=1)
        test_counts += counts
        test_cases.clear(); gc.collect()
    table_test = pd.DataFrame([{"caso": c, **summarize_counts(k)} for c, k in zip(CASES["test"], test_counts)]
                              + [{"caso": "TODOS", **summarize_counts(sum_counts(test_counts))}])
    table_test.to_csv(OUT_3D_DIR / "resultado_test_config_congelada.csv", index=False)
    display(table_test[SUMMARY_COLUMNS].round(3))
else:
    print("Test sin evaluar (EVALUAR_TEST = False).")


## 11. Cómo leer los resultados y qué falta

- **Techo (10.1) vs modelo (10.4/10.5).** La distancia entre ambos es lo que aún se puede ganar mejorando la red; la distancia entre el techo y 1.0 es lo que impone el pipeline (resolución 256, tamaño de los fragmentos).
- **Métricas 2D vs 3D.** Las de la sección 23.6b son por corte; estas son por fragmento en 3D, que es lo que usa el challenge. Las dos se reportan, con su unidad.
- **Límites.** La clase de cada fragmento 3D sale de la mayoría semántica del decoder; la cabeza de pertenencia de Fabián puede reemplazarla con sus probabilidades. El ajuste usa 5 casos. El enlace es corte a corte y no corrige un fragmento que el decoder ya fundió.
- **Pendiente para la entrega:** repetir 10.3–10.7 con el checkpoint con skip connections; evaluar `test` una vez con `config_3d.json` congelado; usar `predict_case_3d` en los visualizadores 2 y 3.
